In [1]:
!pip install pgmpy pygad scikit-learn fairlearn networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 61.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstallin

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from fairlearn.metrics import (demographic_parity_difference, 
                               equalized_odds_difference, 
                               true_positive_rate_difference, 
                               false_positive_rate_difference)
from fairlearn.metrics import MetricFrame
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import sparse
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

In [3]:
def preprocessCompas():
    df=pd.read_csv('/kaggle/input/compas/compas-scores-two-years.csv')
    
    df = df.loc[
    (df['days_b_screening_arrest'] <= 30) &
    (df['days_b_screening_arrest'] >= -30) &
    (df['is_recid'] != -1) &
    (df['c_charge_degree'] != 'O') &
    (df['score_text'] != 'N/A'),
    ['age', 'c_charge_degree', 'race', 'age_cat', 'score_text', 'sex',
     'priors_count', 'days_b_screening_arrest', 'decile_score','juv_other_count','juv_misd_count',
     'juv_fel_count',"c_charge_desc",
     'is_recid', 'two_year_recid', 'c_jail_in', 'c_jail_out']  ]

    #upsampling not needed 
    # print(df['two_year_recid'].value_counts())
    # print(df['is_recid'].value_counts())
    # print(df['sex'].value_counts())
    # print(df['race'].value_counts())
    race_mapping={"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sensitive_race = df["race"].map(race_mapping)
    sex_mapping={"Male":0,"Female":1}
    sensitive_sex=df['sex'].map(sex_mapping)
    df['sex']=sensitive_sex
    df['race']=sensitive_race

    #calculating no of days in jail
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    
    df['jail_time']= (df['c_jail_out'] - df['c_jail_in']).dt.days
    df = df.drop(columns=['c_jail_in', 'c_jail_out'])
    df['jail_time'] = df['jail_time'].fillna(0)

    numerical_features = [
    "age", "priors_count", "days_b_screening_arrest", "decile_score","jail_time",'juv_other_count','juv_misd_count',
     'juv_fel_count'
    ]

    categorical_features = [
    "c_charge_degree", "race", "age_cat", "score_text", "sex","c_charge_desc"
    ]
    numerical_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown="ignore")


    preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
    )
    pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
    ])
    #any one target variable
    X=df.drop(columns=["is_recid","two_year_recid"],axis=1)
    # y=df["is_recid"].values
    y=df["two_year_recid"].values
    X_train, X_test, y_train, y_test, sensitive_race_train, sensitive_race_test, sensitive_sex_train, sensitive_sex_test = train_test_split(
    X, y, sensitive_race,sensitive_sex,test_size=0.2,random_state=42,stratify=y  
    )
    X_train = pipeline.fit_transform(X_train)
    X_test = pipeline.transform(X_test)
    
    
    
    return X_train, X_test, y_train, y_test, sensitive_race_train, sensitive_race_test, sensitive_sex_train, sensitive_sex_test
    
    

    


    

In [4]:
X_train, X_test, y_train, y_test, sensitive_race_train, sensitive_race_test, sensitive_sex_train, sensitive_sex_test = preprocessCompas()

/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)


In [5]:
def four_metrics(y_test, y_pred, sensitive_race_test, sensitive_sex_test):
    race_metrics = {
        "Demographic Parity Difference": demographic_parity_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_race_test),
        "Equalized Odds Difference": equalized_odds_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_race_test),
        "Equal Opportunity Difference": true_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_race_test),
        "False Positive Rate Difference": false_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_race_test)
    }
    sex_metrics = {
        "Demographic Parity Difference": demographic_parity_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_sex_test),
        "Equalized Odds Difference": equalized_odds_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_sex_test),
        "Equal Opportunity Difference": true_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_sex_test),
        "False Positive Rate Difference": false_positive_rate_difference(
            y_true=y_test, y_pred=y_pred, sensitive_features=sensitive_sex_test)
    }
    return race_metrics, sex_metrics

In [6]:
def base_model():
    print("THIS IS BASE MODEL.")
    base_model = LogisticRegression(max_iter=1000,random_state=42)
    base_model.fit(X_train,y_train)
    base_model_pred= base_model.predict(X_test)
    print("Accuracy", accuracy_score(y_test,base_model_pred))
    # print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    # print("Classification Report:\n", classification_report(y_test, y_pred))
    met_race, met_sex = four_metrics(y_test, base_model_pred, sensitive_race_test, sensitive_sex_test)
    print("Fairness metrics for Race:")
    for metric_name, value in met_race.items():
        print(f"{metric_name}: {value:.3f}")
    print("Fairness metrics for Sex:")
    for metric_name, value in met_sex.items():
        print(f"{metric_name}: {value:.3f}")
    return base_model_pred
base_model_pred = base_model()

THIS IS BASE MODEL.
Accuracy 0.691497975708502
Fairness metrics for Race:
Demographic Parity Difference: 0.487
Equalized Odds Difference: 0.665
Equal Opportunity Difference: 0.665
False Positive Rate Difference: 0.289
Fairness metrics for Sex:
Demographic Parity Difference: 0.274
Equalized Odds Difference: 0.306
Equal Opportunity Difference: 0.306
False Positive Rate Difference: 0.192
